# Notebook 3B: Generative Head Experiments (exp3 & exp4)
**Run B:** Requires silver explanations CSV. Attach BOTH:
- `trainmultihate` dataset
- Your silver explanations dataset (`silver_explanations_human.csv`)

- **exp3**: + Generative Head only (no consistency loss)
- **exp4**: Full Model (consistency + generative head)

Estimated runtime: ~9 hours on T4 x2

---
## 1. Environment Setup

In [1]:
!pip install -q datasets transformers accelerate

import os
import gc
import json
import time
import random
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ─── Reproducibility ───
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ─── Paths (Auto-detect) ───
OUTPUT_DIR = '/kaggle/working'
MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Silver explanations path (Phase 3) ───────────────────────────────────────
# Auto-detect: search ALL of /kaggle/input/ for any CSV with 'silver' or 'explanation' in the name
SILVER_PATH = None
_silver_globs = glob.glob('/kaggle/input/**/*.csv', recursive=True)
for _sg in _silver_globs:
    _bn = os.path.basename(_sg).lower()
    if 'silver' in _bn or 'explanation' in _bn:
        SILVER_PATH = _sg
        break
if SILVER_PATH:
    print(f'✅ Silver explanations found: {SILVER_PATH}')
else:
    print('⚠️  No silver explanations CSV found in /kaggle/input/')
    # List all CSVs to help debug
    print(f'   Available CSVs: {_silver_globs}')

# Auto-detect dataset
data_files = glob.glob('/kaggle/input/**/train.json', recursive=True)
if data_files:
    DATA_DIR = os.path.dirname(data_files[0])
    print(f'Found dataset at: {DATA_DIR}')
else:
    DATA_DIR = '/kaggle/input'

TRAIN_PATH = os.path.join(OUTPUT_DIR, 'data', 'train.json')
DEV_PATH = os.path.join(OUTPUT_DIR, 'data', 'dev.json')
TEST_PATH = os.path.join(OUTPUT_DIR, 'data', 'test.json')

# Check if pre-split files exist in the dataset
if os.path.exists(os.path.join(DATA_DIR, 'train.json')) and \
   os.path.exists(os.path.join(DATA_DIR, 'dev.json')) and \
   os.path.exists(os.path.join(DATA_DIR, 'test.json')):
    TRAIN_PATH = os.path.join(DATA_DIR, 'train.json')
    DEV_PATH = os.path.join(DATA_DIR, 'dev.json')
    TEST_PATH = os.path.join(DATA_DIR, 'test.json')
    print('Using pre-split dataset files.')
elif os.path.exists(os.path.join(DATA_DIR, 'train.json')):
    # Only train.json exists — split it 80/10/10
    print('Only train.json found. Splitting into train/dev/test (80/10/10)...')
    os.makedirs(os.path.join(OUTPUT_DIR, 'data'), exist_ok=True)
    
    with open(os.path.join(DATA_DIR, 'train.json'), 'r', encoding='utf-8') as f:
        all_data = json.load(f)
    
    df_all = pd.DataFrame(all_data)
    df_all['type_of_hate'] = df_all['type_of_hate'].fillna('None')
    df_all['target_of_hate'] = df_all['target_of_hate'].fillna('None')
    
    # Stratified split on type_of_hate
    df_train, df_temp = train_test_split(df_all, test_size=0.2, random_state=SEED, 
                                          stratify=df_all['type_of_hate'])
    df_dev, df_test = train_test_split(df_temp, test_size=0.5, random_state=SEED,
                                        stratify=df_temp['type_of_hate'])
    
    df_train.to_json(TRAIN_PATH, orient='records', force_ascii=False, indent=2)
    df_dev.to_json(DEV_PATH, orient='records', force_ascii=False, indent=2)
    df_test.to_json(TEST_PATH, orient='records', force_ascii=False, indent=2)
    
    print(f'  Train: {len(df_train):,} samples')
    print(f'  Dev:   {len(df_dev):,} samples')
    print(f'  Test:  {len(df_test):,} samples')
else:
    # Fallback: download from HuggingFace
    print('No local dataset found. Downloading from HuggingFace...')
    from datasets import load_dataset
    dataset = load_dataset('aridhasan/BanglaMultiHate')
    os.makedirs(os.path.join(OUTPUT_DIR, 'data'), exist_ok=True)
    for split_name in ['train', 'dev', 'test']:
        df = dataset[split_name].to_pandas()
        df['type_of_hate'] = df['type_of_hate'].fillna('None')
        df['target_of_hate'] = df['target_of_hate'].fillna('None')
        p = os.path.join(OUTPUT_DIR, 'data', f'{split_name}.json')
        df.to_json(p, orient='records', force_ascii=False, indent=2)
    TRAIN_PATH = os.path.join(OUTPUT_DIR, 'data', 'train.json')
    DEV_PATH = os.path.join(OUTPUT_DIR, 'data', 'dev.json')
    TEST_PATH = os.path.join(OUTPUT_DIR, 'data', 'test.json')

print(f'\nTrain: {TRAIN_PATH}')
print(f'Dev:   {DEV_PATH}')
print(f'Test:  {TEST_PATH}')
print('\n✅ Environment ready')


Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
✅ Silver explanations found: /kaggle/input/datasets/nahidgazi/silver-explanations-human/silver_explanations_verified.csv
Found dataset at: /kaggle/input/datasets/nahidgazi/trainmultihate
Only train.json found. Splitting into train/dev/test (80/10/10)...
  Train: 28,417 samples
  Dev:   3,552 samples
  Test:  3,553 samples

Train: /kaggle/working/data/train.json
Dev:   /kaggle/working/data/dev.json
Test:  /kaggle/working/data/test.json

✅ Environment ready


---
## 2. Label Definitions & Config

In [2]:
# ═══════════════════════════════════════════════════════════
# CANONICAL LABEL ORDERINGS — MUST MATCH ACROSS ALL CODE
# ═══════════════════════════════════════════════════════════
TYPE_LABELS = ['None', 'Abusive', 'Political Hate', 'Religious Hate', 'Gender Hate']
TARGET_LABELS = ['None', 'Individual', 'Organization', 'Community', 'Society']
SEVERITY_LABELS = ['Little to None', 'Mild', 'Severe']

TYPE2IDX = {label: idx for idx, label in enumerate(TYPE_LABELS)}
TARGET2IDX = {label: idx for idx, label in enumerate(TARGET_LABELS)}
SEVERITY2IDX = {label: idx for idx, label in enumerate(SEVERITY_LABELS)}

NUM_TYPE = len(TYPE_LABELS)      # 5
NUM_TARGET = len(TARGET_LABELS)  # 5
NUM_SEVERITY = len(SEVERITY_LABELS)  # 3

# Consistency loss indices
TYPE_NONE_IDX = 0
TARGET_NONE_IDX = 0
SEV_LITTLE_IDX = 0
SEV_SEVERE_IDX = 2

# Encoder
ENCODER_NAME = 'csebuetnlp/banglabert'
MAX_LENGTH = 256

print(f'Type classes:     {NUM_TYPE} {TYPE_LABELS}')
print(f'Target classes:   {NUM_TARGET} {TARGET_LABELS}')
print(f'Severity classes: {NUM_SEVERITY} {SEVERITY_LABELS}')

Type classes:     5 ['None', 'Abusive', 'Political Hate', 'Religious Hate', 'Gender Hate']
Target classes:   5 ['None', 'Individual', 'Organization', 'Community', 'Society']
Severity classes: 3 ['Little to None', 'Mild', 'Severe']


---
## 3. Dataset Class

In [3]:
class BanglaHateDataset(Dataset):
    """PyTorch Dataset for BanglaMultiHate multi-task classification.
    
    Optionally loads silver explanations for generative head training.
    """
    
    def __init__(self, data_path, tokenizer, max_length=256, silver_path=None):
        # Load classification data
        if data_path.endswith('.json'):
            with open(data_path, 'r', encoding='utf-8') as f:
                raw = json.load(f)
            self.df = pd.DataFrame(raw)
        else:
            self.df = pd.read_csv(data_path)
        
        self.df['type_of_hate'] = self.df['type_of_hate'].fillna('None').astype(str).str.strip()
        # Remap old taxonomy labels
        _LABEL_REMAP = {'Profane': 'Abusive', 'Sexism': 'Gender Hate'}
        self.df['type_of_hate'] = self.df['type_of_hate'].replace(_LABEL_REMAP)
        self.df['target_of_hate'] = self.df['target_of_hate'].fillna('None').astype(str).str.strip()
        self.df['severity_of_hate'] = self.df['severity_of_hate'].astype(str).str.strip()
        
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # ── Load silver explanations if provided ──────────────────────
        self.explanations = {}  # comment -> explanation text
        if silver_path and os.path.exists(silver_path):
            if silver_path.endswith('.csv'):
                silver_df = pd.read_csv(silver_path, encoding='utf-8-sig')
            else:
                with open(silver_path, 'r', encoding='utf-8') as f:
                    silver_df = pd.DataFrame(json.load(f))
            
            # Only use approved / edited entries
            if 'verification_status' in silver_df.columns:
                silver_df = silver_df[silver_df['verification_status'].isin(['approved', 'edited'])]
            
            # Build lookup: comment -> explanation
            for _, row in silver_df.iterrows():
                comment = str(row.get('comment', '')).strip()
                expl = str(row.get('silver_explanation', '')).strip()
                if comment and expl:
                    self.explanations[comment] = expl
            
            print(f'  Loaded {len(self.explanations):,} silver explanations from {os.path.basename(silver_path)}')
        else:
            if silver_path:
                print(f'  WARNING: silver_path not found: {silver_path}')
        
        # Validate labels
        invalid_types = set(self.df['type_of_hate'].unique()) - set(TYPE_LABELS)
        invalid_targets = set(self.df['target_of_hate'].unique()) - set(TARGET_LABELS)
        invalid_sev = set(self.df['severity_of_hate'].unique()) - set(SEVERITY_LABELS)
        if invalid_types: print(f'  WARNING: Unknown type labels: {invalid_types}')
        if invalid_targets: print(f'  WARNING: Unknown target labels: {invalid_targets}')
        if invalid_sev: print(f'  WARNING: Unknown severity labels: {invalid_sev}')
        
        print(f'  Loaded {len(self.df):,} samples from {os.path.basename(data_path)}')
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        comment_text = str(row['comment'])
        
        # Tokenize comment
        encoding = self.tokenizer(
            comment_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'type_label': torch.tensor(TYPE2IDX.get(row['type_of_hate'], 0), dtype=torch.long),
            'target_label': torch.tensor(TARGET2IDX.get(row['target_of_hate'], 0), dtype=torch.long),
            'severity_label': torch.tensor(SEVERITY2IDX.get(row['severity_of_hate'], 0), dtype=torch.long),
        }
        
        # ── Tokenize silver explanation if available ─────────────────
        expl_text = self.explanations.get(comment_text.strip(), '')
        if expl_text:
            expl_enc = self.tokenizer(
                expl_text,
                max_length=64,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            item['expl_input_ids'] = expl_enc['input_ids'].squeeze(0)
            item['expl_attention_mask'] = expl_enc['attention_mask'].squeeze(0)
        else:
            item['expl_input_ids'] = torch.zeros(64, dtype=torch.long)
            item['expl_attention_mask'] = torch.zeros(64, dtype=torch.long)
        
        return item
    
    def get_class_weights(self, task_col, label_order):
        """Inverse-frequency weights: w_c = N / (C * n_c)"""
        counts = self.df[task_col].value_counts()
        total = len(self.df)
        n_classes = len(label_order)
        weights = [total / (n_classes * counts.get(label, 1)) for label in label_order]
        return torch.FloatTensor(weights)


---
## 4. Model Architecture

In [4]:
class ClassificationHead(nn.Module):
    """Two-layer MLP classification head."""
    def __init__(self, input_dim, num_classes, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)


class LSTMDecoder(nn.Module):
    """LSTM-based decoder for explanation generation.
    
    Initialized from the encoder's [CLS] hidden state.
    Uses teacher forcing during training.
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.projection = nn.Linear(hidden_dim, vocab_size)
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        # Project encoder CLS (768) -> LSTM hidden (hidden_dim)
        self.init_h = nn.Linear(768, hidden_dim)
        self.init_c = nn.Linear(768, hidden_dim)
    
    def forward(self, decoder_input_ids, encoder_cls_hidden):
        """
        Args:
            decoder_input_ids: (batch, seq_len) - teacher-forced input
            encoder_cls_hidden: (batch, 768) - CLS representation from encoder
        Returns:
            logits: (batch, seq_len, vocab_size)
        """
        batch_size = decoder_input_ids.size(0)
        
        # Initialize LSTM hidden state from encoder CLS
        h0 = self.init_h(encoder_cls_hidden).unsqueeze(0).expand(self.num_layers, -1, -1).contiguous()
        c0 = self.init_c(encoder_cls_hidden).unsqueeze(0).expand(self.num_layers, -1, -1).contiguous()
        
        # Embed decoder inputs
        embeds = self.embedding(decoder_input_ids)  # (batch, seq_len, embed_dim)
        
        # LSTM forward
        lstm_out, _ = self.lstm(embeds, (h0, c0))  # (batch, seq_len, hidden_dim)
        
        # Project to vocabulary
        logits = self.projection(lstm_out)  # (batch, seq_len, vocab_size)
        
        return logits


class ConsistencyConstrainedMTL(nn.Module):
    """Multi-Task Learning model with consistency constraints.
    
    Architecture:
        BanglaBERT Encoder -> [CLS] -> 3 Classification Heads
                                    -> Optional LSTM Decoder (Gen Head)
    """
    def __init__(self, encoder_name='csebuetnlp/banglabert',
                 num_type=5, num_target=5, num_severity=3,
                 hidden_dim=256, dropout=0.3, use_gen_head=False,
                 gen_hidden_dim=512, gen_num_layers=1):
        super().__init__()
        
        # Shared encoder
        self.encoder = AutoModel.from_pretrained(encoder_name)
        enc_dim = self.encoder.config.hidden_size  # 768
        
        # Classification heads
        self.type_head = ClassificationHead(enc_dim, num_type, hidden_dim, dropout)
        self.target_head = ClassificationHead(enc_dim, num_target, hidden_dim, dropout)
        self.severity_head = ClassificationHead(enc_dim, num_severity, hidden_dim, dropout)
        
        # Optional generative decoder
        self.use_gen_head = use_gen_head
        if use_gen_head:
            vocab_size = self.encoder.config.vocab_size
            self.gen_decoder = LSTMDecoder(
                vocab_size=vocab_size,
                embed_dim=256,
                hidden_dim=gen_hidden_dim,
                num_layers=gen_num_layers,
                dropout=dropout
            )
    
    def forward(self, input_ids, attention_mask, decoder_input_ids=None):
        # Encode
        enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_hidden = enc_out.last_hidden_state[:, 0, :]  # (batch, 768)
        
        # Classification
        type_logits = self.type_head(cls_hidden)
        target_logits = self.target_head(cls_hidden)
        severity_logits = self.severity_head(cls_hidden)
        
        # Generation (optional)
        gen_logits = None
        if self.use_gen_head and decoder_input_ids is not None:
            gen_logits = self.gen_decoder(decoder_input_ids, cls_hidden)
        
        return type_logits, target_logits, severity_logits, gen_logits
    
    def count_parameters(self):
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        return trainable, total

---
## 5. Loss Functions

In [5]:
class FocalLoss(nn.Module):
    """Focal Loss (Lin et al., 2017) for class imbalance."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        if alpha is not None:
            self.register_buffer('alpha', alpha)
        else:
            self.alpha = None
    
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


class ConsistencyPenaltyLoss(nn.Module):
    """Soft differentiable penalty for contradictory multi-task predictions.
    
    Rules from BanglaMultiHate:
      If type=None -> target MUST be None AND severity MUST be Little to None
    
    L_consist = mean(v1 + v2 + 2*v3) where:
      v1 = P(type=None) * P(target!=None)
      v2 = P(type=None) * P(severity!=Little)
      v3 = P(type=None) * P(severity=Severe)  [extra penalty]
    """
    def __init__(self, type_none_idx=0, target_none_idx=0, 
                 sev_little_idx=0, sev_severe_idx=2):
        super().__init__()
        self.type_none_idx = type_none_idx
        self.target_none_idx = target_none_idx
        self.sev_little_idx = sev_little_idx
        self.sev_severe_idx = sev_severe_idx
    
    def forward(self, type_logits, target_logits, severity_logits):
        p_type = F.softmax(type_logits, dim=-1)
        p_target = F.softmax(target_logits, dim=-1)
        p_sev = F.softmax(severity_logits, dim=-1)
        
        p_type_none = p_type[:, self.type_none_idx]
        p_target_has = 1.0 - p_target[:, self.target_none_idx]
        p_sev_has = 1.0 - p_sev[:, self.sev_little_idx]
        p_sev_severe = p_sev[:, self.sev_severe_idx]
        
        v1 = p_type_none * p_target_has
        v2 = p_type_none * p_sev_has
        v3 = p_type_none * p_sev_severe
        
        return (v1 + v2 + 2.0 * v3).mean()

---
## 6. Evaluation Functions

In [6]:
def compute_consistency_violations(type_preds, target_preds, sev_preds):
    """Count predictions where type=None but target!=None or severity!=Little.
    
    Returns:
        violation_count: number of violating samples
        violation_rate: fraction of total samples that violate
    """
    type_none_mask = (type_preds == TYPE_NONE_IDX)
    target_not_none = (target_preds != TARGET_NONE_IDX)
    sev_not_little = (sev_preds != SEV_LITTLE_IDX)
    
    violations = type_none_mask & (target_not_none | sev_not_little)
    count = violations.sum().item()
    rate = count / len(type_preds) if len(type_preds) > 0 else 0
    return count, rate


@torch.no_grad()
def evaluate(model, dataloader, device):
    """Evaluate model on a dataloader.
    
    Returns dict with:
        - type_f1, target_f1, sev_f1 (macro)
        - avg_f1 (mean of 3 macro F1s)
        - cvr (consistency violation rate)
        - per-class F1 for each task
    """
    model.eval()
    all_type_preds, all_target_preds, all_sev_preds = [], [], []
    all_type_labels, all_target_labels, all_sev_labels = [], [], []
    
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        type_logits, target_logits, sev_logits, _ = model(input_ids, attention_mask)
        
        all_type_preds.append(type_logits.argmax(dim=-1).cpu())
        all_target_preds.append(target_logits.argmax(dim=-1).cpu())
        all_sev_preds.append(sev_logits.argmax(dim=-1).cpu())
        all_type_labels.append(batch['type_label'])
        all_target_labels.append(batch['target_label'])
        all_sev_labels.append(batch['severity_label'])
    
    type_preds = torch.cat(all_type_preds)
    target_preds = torch.cat(all_target_preds)
    sev_preds = torch.cat(all_sev_preds)
    type_labels = torch.cat(all_type_labels)
    target_labels = torch.cat(all_target_labels)
    sev_labels = torch.cat(all_sev_labels)
    
    # Macro F1 scores
    type_f1 = f1_score(type_labels, type_preds, average='macro', zero_division=0)
    target_f1 = f1_score(target_labels, target_preds, average='macro', zero_division=0)
    sev_f1 = f1_score(sev_labels, sev_preds, average='macro', zero_division=0)
    avg_f1 = (type_f1 + target_f1 + sev_f1) / 3
    
    # Consistency Violation Rate
    cv_count, cvr = compute_consistency_violations(type_preds, target_preds, sev_preds)
    
    # Per-class F1
    type_per_class = f1_score(type_labels, type_preds, average=None, zero_division=0)
    target_per_class = f1_score(target_labels, target_preds, average=None, zero_division=0)
    sev_per_class = f1_score(sev_labels, sev_preds, average=None, zero_division=0)
    
    return {
        'type_f1': type_f1,
        'target_f1': target_f1,
        'sev_f1': sev_f1,
        'avg_f1': avg_f1,
        'cvr': cvr,
        'cv_count': cv_count,
        'total_samples': len(type_preds),
        'type_per_class': {TYPE_LABELS[i]: float(f) for i, f in enumerate(type_per_class)},
        'target_per_class': {TARGET_LABELS[i]: float(f) for i, f in enumerate(target_per_class)},
        'sev_per_class': {SEVERITY_LABELS[i]: float(f) for i, f in enumerate(sev_per_class)},
        'type_preds': type_preds,
        'target_preds': target_preds,
        'sev_preds': sev_preds,
        'type_labels': type_labels,
        'target_labels': target_labels,
        'sev_labels': sev_labels,
    }

---
## 7. Training Loop

In [7]:
def train_experiment(config):
    """Run a single training experiment.
    
    Args:
        config: dict with experiment configuration
    
    Returns:
        model: trained model
        history: training history dict
        best_metrics: best validation metrics
    """
    print(f'\n{"="*70}')
    print(f'EXPERIMENT: {config["name"]}')
    print(f'  Consistency: {config["use_consistency"]} | Gen Head: {config["use_gen"]}')
    print(f'{"="*70}')
    
    # ── Load tokenizer ──
    tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
    
    # ── Create datasets ──
    silver_path = config.get('silver_path', None) if config.get('use_gen', False) else None
    train_dataset = BanglaHateDataset(TRAIN_PATH, tokenizer, MAX_LENGTH, silver_path=silver_path)
    dev_dataset = BanglaHateDataset(DEV_PATH, tokenizer, MAX_LENGTH)
    
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], 
                              shuffle=True, num_workers=2, pin_memory=True)
    dev_loader = DataLoader(dev_dataset, batch_size=config['batch_size'] * 2,
                            shuffle=False, num_workers=2, pin_memory=True)
    
    # ── Create model ──
    model = ConsistencyConstrainedMTL(
        encoder_name=ENCODER_NAME,
        num_type=NUM_TYPE, num_target=NUM_TARGET, num_severity=NUM_SEVERITY,
        hidden_dim=256, dropout=0.3,
        use_gen_head=config['use_gen']
    ).to(DEVICE)
    
    trainable, total = model.count_parameters()
    print(f'  Parameters: {trainable:,} trainable / {total:,} total')
    
    # ── Loss functions ──
    type_weights = train_dataset.get_class_weights('type_of_hate', TYPE_LABELS).to(DEVICE)
    target_weights = train_dataset.get_class_weights('target_of_hate', TARGET_LABELS).to(DEVICE)
    sev_weights = train_dataset.get_class_weights('severity_of_hate', SEVERITY_LABELS).to(DEVICE)
    
    type_loss_fn = FocalLoss(alpha=type_weights, gamma=2.0)
    target_loss_fn = FocalLoss(alpha=target_weights, gamma=2.0)
    sev_loss_fn = FocalLoss(alpha=sev_weights, gamma=2.0)
    consist_loss_fn = ConsistencyPenaltyLoss(
        type_none_idx=TYPE_NONE_IDX, target_none_idx=TARGET_NONE_IDX,
        sev_little_idx=SEV_LITTLE_IDX, sev_severe_idx=SEV_SEVERE_IDX
    )
    
    # ── Optimizer with differential learning rates ──
    encoder_params = list(model.encoder.parameters())
    head_params = (list(model.type_head.parameters()) + 
                   list(model.target_head.parameters()) + 
                   list(model.severity_head.parameters()))
    
    param_groups = [
        {'params': encoder_params, 'lr': config['encoder_lr']},
        {'params': head_params, 'lr': config['head_lr']},
    ]
    
    if config['use_gen']:
        gen_params = list(model.gen_decoder.parameters())
        param_groups.append({'params': gen_params, 'lr': config['head_lr']})
    
    optimizer = AdamW(param_groups, weight_decay=0.01)
    
    total_steps = len(train_loader) * config['epochs']
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    
    # ── Training ──
    best_val_f1 = 0.0
    best_metrics = None
    patience_counter = 0
    history = {'train_loss': [], 'val_metrics': []}
    
    for epoch in range(config['epochs']):
        model.train()
        epoch_loss = 0.0
        epoch_losses = {'type': 0, 'target': 0, 'sev': 0, 'consist': 0, 'total': 0}
        
        # Warmup lambda for consistency loss
        lambda_c = config['lambda_start'] + (
            config['lambda_end'] - config['lambda_start']
        ) * min(epoch / max(config['epochs'] - 1, 1), 1.0)
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config["epochs"]}', leave=True)
        
        for batch in pbar:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            type_labels = batch['type_label'].to(DEVICE)
            target_labels = batch['target_label'].to(DEVICE)
            sev_labels = batch['severity_label'].to(DEVICE)
            
            # Forward
            type_logits, target_logits, sev_logits, gen_logits = model(
                input_ids, attention_mask
            )
            
            # Classification losses
            l_type = type_loss_fn(type_logits, type_labels)
            l_target = target_loss_fn(target_logits, target_labels)
            l_sev = sev_loss_fn(sev_logits, sev_labels)
            
            loss = l_type + l_target + l_sev
            
            # Consistency penalty
            if config['use_consistency']:
                l_consist = consist_loss_fn(type_logits, target_logits, sev_logits)
                loss = loss + lambda_c * l_consist
                epoch_losses['consist'] += l_consist.item()
            
            # Backward
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            epoch_losses['type'] += l_type.item()
            epoch_losses['target'] += l_target.item()
            epoch_losses['sev'] += l_sev.item()
            epoch_losses['total'] += loss.item()
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        # Average losses
        n_batches = len(train_loader)
        for k in epoch_losses:
            epoch_losses[k] /= n_batches
        
        # ── Validation ──
        val_metrics = evaluate(model, dev_loader, DEVICE)
        
        print(f'  Train Loss: {epoch_losses["total"]:.4f} '
              f'(type={epoch_losses["type"]:.4f}, target={epoch_losses["target"]:.4f}, '
              f'sev={epoch_losses["sev"]:.4f}, consist={epoch_losses["consist"]:.4f})')
        print(f'  Val   F1:   type={val_metrics["type_f1"]:.4f}, '
              f'target={val_metrics["target_f1"]:.4f}, sev={val_metrics["sev_f1"]:.4f}, '
              f'avg={val_metrics["avg_f1"]:.4f} | CVR={val_metrics["cvr"]*100:.2f}%'
              f' ({val_metrics["cv_count"]}/{val_metrics["total_samples"]})')
        
        # Save history
        history['train_loss'].append(epoch_losses)
        # Remove tensors before saving to JSON
        val_save = {k: v for k, v in val_metrics.items() 
                    if not isinstance(v, torch.Tensor)}
        history['val_metrics'].append(val_save)
        
        # ── Best model checkpoint ──
        if val_metrics['avg_f1'] > best_val_f1:
            best_val_f1 = val_metrics['avg_f1']
            best_metrics = val_metrics
            patience_counter = 0
            
            checkpoint_path = os.path.join(MODEL_DIR, f'{config["name"]}_best.pt')
            torch.save({
                'model_state_dict': model.state_dict(),
                'config': config,
                'epoch': epoch,
                'best_val_f1': best_val_f1,
                'val_metrics': val_save,
            }, checkpoint_path)
            print(f'  ✅ New best model saved! Avg F1: {best_val_f1:.4f}')
        else:
            patience_counter += 1
            if patience_counter >= config.get('patience', 5):
                print(f'  ⚠️ Early stopping at epoch {epoch+1} (patience={config["patience"]})')
                break
    
    # Save training history
    history_path = os.path.join(RESULTS_DIR, f'{config["name"]}_history.json')
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)
    
    print(f'\n  🏁 {config["name"]} complete. Best Avg F1: {best_val_f1:.4f}')
    
    # Clean up VRAM
    del model, optimizer, scheduler, train_loader, dev_loader
    gc.collect()
    torch.cuda.empty_cache()
    
    return history, best_metrics

---
## 8. Smoke Test (Verify Pipeline on 100 Samples)

In [8]:
print('🧪 Running smoke test (100 samples, 2 epochs)...')

tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
smoke_dataset = BanglaHateDataset(TRAIN_PATH, tokenizer, MAX_LENGTH)

# Use only first 100 samples
smoke_dataset.df = smoke_dataset.df.head(100)
smoke_loader = DataLoader(smoke_dataset, batch_size=8, shuffle=True)

# Quick model test
smoke_model = ConsistencyConstrainedMTL(
    encoder_name=ENCODER_NAME,
    use_gen_head=False
).to(DEVICE)

consist_fn = ConsistencyPenaltyLoss()
optimizer = AdamW(smoke_model.parameters(), lr=2e-5)

smoke_model.train()
losses = []
for epoch in range(2):
    epoch_loss = 0
    for batch in smoke_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        
        t_out, tg_out, s_out, _ = smoke_model(input_ids, attention_mask)
        
        l = (F.cross_entropy(t_out, batch['type_label'].to(DEVICE)) +
             F.cross_entropy(tg_out, batch['target_label'].to(DEVICE)) +
             F.cross_entropy(s_out, batch['severity_label'].to(DEVICE)) +
             0.1 * consist_fn(t_out, tg_out, s_out))
        
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
        epoch_loss += l.item()
    
    avg_loss = epoch_loss / len(smoke_loader)
    losses.append(avg_loss)
    print(f'  Epoch {epoch+1}: Loss = {avg_loss:.4f}')

assert losses[1] < losses[0], f'Loss did not decrease: {losses}'
print('✅ Smoke test PASSED — loss is decreasing')

# Cleanup
del smoke_model, smoke_dataset, smoke_loader, optimizer
gc.collect()
torch.cuda.empty_cache()

🧪 Running smoke test (100 samples, 2 epochs)...


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
electra.embeddings.position_ids                   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

  Epoch 1: Loss = 4.0867
  Epoch 2: Loss = 3.7028
✅ Smoke test PASSED — loss is decreasing


### Experiment 3: + Generative Head (No Consistency)

In [9]:
# Silver explanations MUST be attached for this notebook
assert SILVER_PATH is not None, 'ERROR: Silver explanations CSV not found! Attach your silver dataset to this notebook.'
print(f'Silver explanations: {SILVER_PATH}')

exp3_config = {
    'name': 'exp3_generative',
    'use_consistency': False,
    'use_gen': True,
    'batch_size': 16,
    'epochs': 10,
    'encoder_lr': 2e-5,
    'head_lr': 1e-4,
    'lambda_start': 0.0,
    'lambda_end': 0.0,
    'gen_weight': 0.5,
    'patience': 5,
    'silver_path': SILVER_PATH,
}
exp3_history, exp3_metrics = train_experiment(exp3_config)


Silver explanations: /kaggle/input/datasets/nahidgazi/silver-explanations-human/silver_explanations_verified.csv

EXPERIMENT: exp3_generative
  Consistency: False | Gen Head: True
  Loaded 2,000 silver explanations from silver_explanations_verified.csv
  Loaded 28,417 samples from train.json
  Loaded 3,552 samples from dev.json


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
electra.embeddings.position_ids                   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Parameters: 137,593,101 trainable / 137,593,101 total


Epoch 1/10:   0%|          | 0/1777 [00:00<?, ?it/s]

...

Epoch 9/10:   0%|          | 0/1777 [00:00<?, ?it/s]

  Train Loss: 0.2623 (type=0.0585, target=0.0863, sev=0.1174, consist=0.0000)
  Val   F1:   type=0.5064, target=0.5217, sev=0.5882, avg=0.5388 | CVR=5.60% (199/3552)
  ✅ New best model saved! Avg F1: 0.5388


Epoch 10/10:   0%|          | 0/1777 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79d5e79ae8e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79d5e79ae8e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  Train Loss: 0.2184 (type=0.0525, target=0.0662, sev=0.0998, consist=0.0000)
  Val   F1:   type=0.5054, target=0.5240, sev=0.5887, avg=0.5393 | CVR=4.81% (171/3552)
  ✅ New best model saved! Avg F1: 0.5393

  🏁 exp3_generative complete. Best Avg F1: 0.5393


### Experiment 4: Full Model (Consistency + Gen Head)

In [10]:
exp4_config = {
    'name': 'exp4_full',
    'use_consistency': True,
    'use_gen': True,
    'batch_size': 16,
    'epochs': 10,
    'encoder_lr': 2e-5,
    'head_lr': 1e-4,
    'lambda_start': 0.01,
    'lambda_end': 1.0,
    'gen_weight': 0.5,
    'patience': 5,
    'silver_path': SILVER_PATH,
}
exp4_history, exp4_metrics = train_experiment(exp4_config)



EXPERIMENT: exp4_full
  Consistency: True | Gen Head: True
  Loaded 2,000 silver explanations from silver_explanations_verified.csv
  Loaded 28,417 samples from train.json
  Loaded 3,552 samples from dev.json


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
electra.embeddings.position_ids                   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Parameters: 137,593,101 trainable / 137,593,101 total


Epoch 1/10:   0%|          | 0/1777 [00:00<?, ?it/s]

...

Epoch 9/10:   0%|          | 0/1777 [00:00<?, ?it/s]

  Train Loss: 0.3396 (type=0.0836, target=0.0989, sev=0.1354, consist=0.0243)
  Val   F1:   type=0.5108, target=0.5513, sev=0.6051, avg=0.5557 | CVR=0.00% (0/3552)


Epoch 10/10:   0%|          | 0/1777 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79d5e79ae8e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79d5e79ae8e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  Train Loss: 0.2850 (type=0.0740, target=0.0802, sev=0.1119, consist=0.0189)
  Val   F1:   type=0.5148, target=0.5518, sev=0.6013, avg=0.5560 | CVR=0.00% (0/3552)

  🏁 exp4_full complete. Best Avg F1: 0.5572


In [11]:
print('\n' + '=' * 70)
print('RUN B COMPLETE: exp3 (Generative) + exp4 (Full Model)')
print('=' * 70)
print(f'\nexp3 Best Avg F1: {exp3_metrics["avg_f1"]:.4f} | CVR: {exp3_metrics["cvr"]*100:.2f}%')
print(f'exp4 Best Avg F1: {exp4_metrics["avg_f1"]:.4f} | CVR: {exp4_metrics["cvr"]*100:.2f}%')
print(f'\nModels saved to: {MODEL_DIR}/')
print(f'Results saved to: {RESULTS_DIR}/')



RUN B COMPLETE: exp3 (Generative) + exp4 (Full Model)

exp3 Best Avg F1: 0.5393 | CVR: 4.81%
exp4 Best Avg F1: 0.5572 | CVR: 0.00%

Models saved to: /kaggle/working/models/
Results saved to: /kaggle/working/results/
